In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
import numpy as np
from world_cup_2026.config import RAW_DATA_DIR
from world_cup_2026.data_ingestion.normalize import normalize_dataframe_teams
from world_cup_2026.features.elo import calculate_elo
from world_cup_2026.features.form import FormCalculator

df_results = pd.read_csv(RAW_DATA_DIR / "martj42_results" / "results.csv", parse_dates=["date"])
df_results_norm = normalize_dataframe_teams(df_results, ["home_team", "away_team"])
df_elo = calculate_elo(df_results_norm)

calc = FormCalculator(df_results_norm, windows=[5, 10, 20])

print("Setup complete.")
print(f"Last match in dataset: {df_results_norm['date'].max().date()}")

2026-03-31 15:18:57.718 | INFO     | world_cup_2026.config:<module>:12 - PROJ_ROOT path is: C:\Users\feder\Documents\data_repos\world-cup-2026-predictor
2026-03-31 15:18:58.580 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Guernsey' — add to normalize.py if needed.
2026-03-31 15:18:58.583 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-03-31 15:18:58.588 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Alderney' — add to normalize.py if needed.
2026-03-31 15:18:58.595 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-03-31 15:18:58.602 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Basque Country' — add to normalize.py if needed.
2026-03-31 1

Setup complete.
Last match in dataset: 2026-01-26


In [4]:
print(df_elo.columns.tolist())
print(df_elo.tail(3))

['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral', 'elo_pre_home', 'elo_pre_away', 'elo_diff', 'elo_post_home', 'elo_post_away', 'win_prob_home']
            date   home_team away_team  home_score  away_score tournament  \
49068 2026-01-22      Panama    Mexico           0           1   Friendly   
49069 2026-01-25     Bolivia    Mexico           0           1   Friendly   
49070 2026-01-26  Uzbekistan     China           2           2   Friendly   

              city               country  neutral  elo_pre_home  elo_pre_away  \
49068  Panama City                Panama    False   1817.339796   1899.729447   
49069   Santa Cruz               Bolivia    False   1722.287912   1907.401555   
49070        Dubai  United Arab Emirates     True   1852.434988   1545.870021   

         elo_diff  elo_post_home  elo_post_away  win_prob_home  
49068  -82.389651    1809.667689    1907.401555       0.383605  
49069 -185.113643    1717.163097  

In [5]:
AS_OF = pd.Timestamp("2026-03-31")

def get_team_elo(team: str, as_of: pd.Timestamp) -> float:
    """Get most recent Elo for a team strictly before as_of date."""
    mask = (
        ((df_elo["home_team"] == team) | (df_elo["away_team"] == team))
        & (df_elo["date"] < as_of)
    )
    team_matches = df_elo[mask].sort_values("date")
    if team_matches.empty:
        return 1500.0
    last = team_matches.iloc[-1]
    if last["home_team"] == team:
        return last["elo_pre_home"]
    else:
        return last["elo_pre_away"]


def predict_match(team_a: str, team_b: str, as_of: pd.Timestamp) -> dict:
    """
    Predict match outcome using Elo + recent form.
    Both teams treated as neutral venue (playoff finals are neutral or near-neutral).
    """
    elo_a = get_team_elo(team_a, as_of)
    elo_b = get_team_elo(team_b, as_of)
    elo_diff = elo_a - elo_b

    # Elo win probability
    p_a_win_elo = 1 / (1 + 10 ** (-elo_diff / 400))
    p_b_win_elo = 1 - p_a_win_elo

    # Draw prior for high-stakes neutral venue matches (historical ~22%)
    DRAW_PRIOR = 0.22
    p_a_win = p_a_win_elo * (1 - DRAW_PRIOR)
    p_b_win = p_b_win_elo * (1 - DRAW_PRIOR)
    p_draw  = DRAW_PRIOR

    # Recent form last 10
    form_a = calc.get_team_current_form(team_a, as_of=as_of, window=10)
    form_b = calc.get_team_current_form(team_b, as_of=as_of, window=10)

    form_win_a = form_a[f"form_10_win_rate"]
    form_win_b = form_b[f"form_10_win_rate"]

    # Form adjustment — small nudge, Elo is the primary signal
    # If form win rate > 0.5, slight boost; if < 0.5, slight penalty
    FORM_WEIGHT = 0.08
    form_adj = (form_win_a - form_win_b) * FORM_WEIGHT
    p_a_win_adj = np.clip(p_a_win + form_adj, 0.05, 0.90)
    p_b_win_adj = np.clip(p_b_win - form_adj, 0.05, 0.90)

    # Renormalize to sum to 1
    total = p_a_win_adj + p_draw + p_b_win_adj
    p_a_win_adj /= total
    p_b_win_adj /= total
    p_draw_norm  = p_draw / total

    return {
        "team_a": team_a,
        "team_b": team_b,
        "elo_a": round(elo_a, 0),
        "elo_b": round(elo_b, 0),
        "elo_diff": round(elo_diff, 0),
        "form_10_win_a": round(form_win_a, 2),
        "form_10_win_b": round(form_win_b, 2),
        "p_a_win": round(p_a_win_adj, 3),
        "p_draw":  round(p_draw_norm, 3),
        "p_b_win": round(p_b_win_adj, 3),
        "predicted_winner": team_a if p_a_win_adj > p_b_win_adj else team_b,
        "confidence": round(max(p_a_win_adj, p_b_win_adj), 3),
    }

print("predict_match() ready.")

predict_match() ready.


In [6]:
matchups = [
    ("Italy",   "Bosnia-Herzegovina",  "UEFA Playoff A — Group B"),
    ("Sweden",  "Poland",              "UEFA Playoff B — Group F"),
    ("Kosovo",  "Turkey",              "UEFA Playoff C — Group D"),
    ("Czechia", "Denmark",             "UEFA Playoff D — Group A"),
    ("Jamaica", "DR Congo",            "FIFA Intercontinental 1 — Group K"),
    ("Bolivia", "Iraq",                "FIFA Intercontinental 2 — Group I"),
]

results = []
for team_a, team_b, label in matchups:
    pred = predict_match(team_a, team_b, AS_OF)
    pred["match"] = label
    results.append(pred)

df_pred = pd.DataFrame(results)[[
    "match", "team_a", "team_b",
    "elo_a", "elo_b", "elo_diff",
    "form_10_win_a", "form_10_win_b",
    "p_a_win", "p_draw", "p_b_win",
    "predicted_winner", "confidence"
]]

pd.set_option("display.max_colwidth", 40)
pd.set_option("display.float_format", "{:.3f}".format)
print(df_pred.to_string(index=False))

                            match  team_a             team_b    elo_a    elo_b  elo_diff  form_10_win_a  form_10_win_b  p_a_win  p_draw  p_b_win predicted_winner  confidence
         UEFA Playoff A — Group B   Italy Bosnia-Herzegovina 1953.000 1623.000   330.000          0.600          0.600    0.679   0.220    0.101            Italy       0.679
         UEFA Playoff B — Group F  Sweden             Poland 1713.000 1810.000   -98.000          0.300          0.700    0.251   0.220    0.529           Poland       0.529
         UEFA Playoff C — Group D  Kosovo             Turkey 1750.000 1907.000  -157.000          0.700          0.700    0.225   0.220    0.555           Turkey       0.555
         UEFA Playoff D — Group A Czechia            Denmark 1776.000 1941.000  -165.000          0.600          0.600    0.217   0.220    0.563          Denmark       0.563
FIFA Intercontinental 1 — Group K Jamaica           DR Congo 1664.000 1728.000   -64.000          0.500          0.600    0.311   

In [7]:
print("\n=== PLAYOFF PREDICTIONS — March 31, 2026 ===\n")
for _, row in df_pred.iterrows():
    print(f"{row['match']}")
    print(f"  {row['team_a']:<22} {row['p_a_win']*100:>5.1f}%  |  draw {row['p_draw']*100:.1f}%  |  {row['p_b_win']*100:>5.1f}%  {row['team_b']}")
    print(f"  Elo: {row['elo_a']:.0f} vs {row['elo_b']:.0f}   Form(10): {row['form_10_win_a']:.0%} vs {row['form_10_win_b']:.0%}")
    print(f"  → Predicted winner: {row['predicted_winner']} ({row['confidence']*100:.1f}% win prob)\n")


=== PLAYOFF PREDICTIONS — March 31, 2026 ===

UEFA Playoff A — Group B
  Italy                   67.9%  |  draw 22.0%  |   10.1%  Bosnia-Herzegovina
  Elo: 1953 vs 1623   Form(10): 60% vs 60%
  → Predicted winner: Italy (67.9% win prob)

UEFA Playoff B — Group F
  Sweden                  25.1%  |  draw 22.0%  |   52.9%  Poland
  Elo: 1713 vs 1810   Form(10): 30% vs 70%
  → Predicted winner: Poland (52.9% win prob)

UEFA Playoff C — Group D
  Kosovo                  22.5%  |  draw 22.0%  |   55.5%  Turkey
  Elo: 1750 vs 1907   Form(10): 70% vs 70%
  → Predicted winner: Turkey (55.5% win prob)

UEFA Playoff D — Group A
  Czechia                 21.7%  |  draw 22.0%  |   56.3%  Denmark
  Elo: 1776 vs 1941   Form(10): 60% vs 60%
  → Predicted winner: Denmark (56.3% win prob)

FIFA Intercontinental 1 — Group K
  Jamaica                 31.1%  |  draw 22.0%  |   46.9%  DR Congo
  Elo: 1664 vs 1728   Form(10): 50% vs 60%
  → Predicted winner: DR Congo (46.9% win prob)

FIFA Intercontinental 

In [8]:
df_pred.to_csv(
    "../../outputs/predictions/playoff_predictions_2026_03_31.csv",
    index=False
)
print("Saved.")

Saved.
